# Get to Know a Dataset: Gramscii European Open Data Catalogue

This notebook is a guided tour of the [Gramscii European Open Data Catalogue](https://registry.opendata.aws/gramscii-european-open-data-catalogue) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The catalogue describes more than half a million public datasets from over 40 European and international providers: national statistical offices, Eurostat, the OECD, the ILO, the ECB, national open data portals, Italian cities and regions, and Earth-observation and marine catalogues. For every dataset it records what the dataset is, how it is structured, which codes and labels it uses in each published language, where it can be read, and under which terms. It holds metadata only: the values stay with the original provider.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket has one top-level prefix, `catalogue/`, which holds:

1. `manifest.json`, which describes the release: when the snapshot was taken, the archive it was written from, every table's rows and columns, and every file's size and SHA-256.
2. One prefix per table, named after the catalogue's database table: `opendata_catalog/`, `opendata_structures/`, `opendata_terms/` and so on.
3. Inside every table keyed by provider, one Hive-style partition per provider, `provider=<id>/`, holding that provider's rows in `data_0.parquet`. The verbatim provider responses in `opendata_native_objects/` are one file, addressed by their SHA-256.

Each release replaces the previous one, so `manifest.json` is the place to start.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.40
# duckdb >= 1.5
# pandas >= 2.3
# matplotlib >= 3.10

# Import the libraries required for this notebook
# Built-ins
import json
from pprint import pprint
# Installed libraries
import boto3
import duckdb
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

First we define the location of the dataset, create an S3 client that does not sign its requests (the bucket is public, so no AWS account is needed), and list the top-level prefix.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "gramscii-european-open-data-catalogue"
region = "eu-central-1"

s3 = boto3.client("s3", region_name=region, config=Config(signature_version=UNSIGNED))

# Print the top-level prefixes
for item in s3.list_objects_v2(Bucket=bucket, Delimiter="/")["CommonPrefixes"]:
    print(item["Prefix"])

Inside `catalogue/` there is one prefix per table, beside the manifest.

In [ ]:
# List the tables of the catalogue
for item in s3.list_objects_v2(Bucket=bucket, Prefix="catalogue/", Delimiter="/")["CommonPrefixes"]:
    print(item["Prefix"])

The manifest says when the snapshot was taken, how many rows each table holds, and which columns it has.

In [ ]:
# Read the release manifest
with s3.get_object(Bucket=bucket, Key="catalogue/manifest.json")["Body"] as body:
    manifest = json.load(body)

print("Snapshot taken at", manifest["taken_at"])
for name, table in manifest["tables"].items():
    print(f"{name:28} {table['rows']:>10,} rows, partitioned by {table['partition']}")

In [ ]:
# Each table keyed by provider holds one partition per provider
for item in s3.list_objects_v2(Bucket=bucket, Prefix="catalogue/opendata_catalog/", Delimiter="/")["CommonPrefixes"][:10]:
    print(item["Prefix"])

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how to work with these data types?

Every table is stored as [Apache Parquet](https://parquet.apache.org/), compressed with Zstandard, and partitioned the Hive way by `provider`. We chose Parquet because it is columnar: a reader that needs three columns of one provider reads only those bytes, over HTTP, without downloading the catalogue. Most tools read it directly: [DuckDB](https://duckdb.org), PyArrow and pandas, Apache Spark, and [Amazon Athena](https://aws.amazon.com/athena/).

Columns keep the types of the catalogue's database: text, booleans, integers, dates, timestamps (stored as UTC instants), lists of text, binary, and JSON. The JSON columns (`names`, `sources`, `dimensions`, `spatial` and others) hold nested metadata exactly as each provider described it, so their shape differs between providers: read them with JSON functions rather than expecting one schema.

In this notebook we use DuckDB, which reads the Parquet files straight from S3 and answers SQL. The `httpfs` extension gives it S3 access; the bucket is public, so no credentials are needed.

In [ ]:
# A DuckDB connection that reads the bucket anonymously
con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute(f"SET s3_region = '{region}'")

root = f"s3://{bucket}/catalogue"

def table(name):
    """The SQL source of one catalogue table, with its provider partitions as a column."""
    return f"read_parquet('{root}/{name}/*/*.parquet', hive_partitioning = true)"

In [ ]:
# The columns and types of the main table
con.sql(f"DESCRIBE SELECT * FROM {table('opendata_catalog')}").df()

### Q: Can you show us an example of downloading and loading data from your dataset?

Let us load the catalogue entries of one provider, ISTAT, the Italian national statistical institute. Because the table is partitioned by provider, DuckDB reads only the `provider=istat` files.

In [ ]:
istat = con.sql(f"""
    SELECT dataset_id, title, title_en, period_start, period_end, licence
    FROM {table('opendata_catalog')}
    WHERE provider = 'istat'
    ORDER BY dataset_id
""").df()

print(len(istat), "ISTAT datasets")
istat.head(10)

One dataset's structure tells which dimensions its observations are classified by. Each dimension draws its codes from a code list, and `opendata_terms` names every code in each published language. Here we take the first ISTAT dataset with a structure and list its dimensions.

In [ ]:
dataset = con.sql(f"""
    SELECT c.dataset_id, c.title, c.structure_id
    FROM {table('opendata_catalog')} c
    WHERE c.provider = 'istat' AND c.structure_available
    ORDER BY c.dataset_id
    LIMIT 1
""").fetchone()
print(dataset)

con.sql(f"""
    SELECT dimension_id, concept_id, codelist_scope
    FROM {table('opendata_structure_dims')}
    WHERE provider = 'istat' AND structure_id = $structure
    ORDER BY dimension_id
""", params={"structure": dataset[2]}).df()

In [ ]:
# The labels of the dimensions' codes, in Italian and English (a code list's own name has an empty code)
con.sql(f"""
    SELECT d.dimension_id, t.code, t.language, t.name
    FROM {table('opendata_structure_dims')} d
    JOIN {table('opendata_terms')} t
      ON t.provider = d.provider AND t.scope = d.codelist_scope
    WHERE d.provider = 'istat' AND d.structure_id = $structure AND t.code <> ''
    ORDER BY d.dimension_id, t.code, t.language
    LIMIT 20
""", params={"structure": dataset[2]}).df()

Every dataset also has a searchable description in each of its languages, in `opendata_documents`. It is plain text, written from the provider's own metadata, and is the text a search index is built from.

In [ ]:
document = con.sql(f"""
    SELECT language, text
    FROM {table('opendata_documents')}
    WHERE provider = 'istat' AND dataset_id = $dataset
""", params={"dataset": dataset[0]}).fetchall()

for language, text in document:
    print(f"--- {language}")
    print(text[:1200])

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

How many datasets does each provider contribute? Parquet lets DuckDB answer by reading only the `provider` partition values and row counts, a few kilobytes per file.

In [ ]:
per_provider = con.sql(f"""
    SELECT provider, count(*) AS datasets
    FROM {table('opendata_catalog')}
    GROUP BY provider
    ORDER BY datasets
""").df()

plt.figure(figsize=(10, 12), dpi=100, facecolor="white")
plt.barh(per_provider["provider"], per_provider["datasets"], color="#3498db")
plt.xscale("log")
plt.title("Datasets per provider", fontsize=16, pad=20, fontweight="bold")
plt.xlabel("Datasets (log scale)", fontsize=12, labelpad=10)
plt.grid(True, axis="x", linestyle="--", alpha=0.3, color="gray")
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

The catalogue is multilingual: every dataset is described in the languages its provider publishes. The next chart counts the searchable descriptions per language.

In [ ]:
per_language = con.sql(f"""
    SELECT language, count(*) AS documents
    FROM {table('opendata_documents')}
    GROUP BY language
    ORDER BY documents DESC
""").df()

plt.figure(figsize=(8, 5), dpi=100, facecolor="white")
plt.bar(per_language["language"], per_language["documents"], color="#2ecc71")
plt.title("Searchable descriptions per language", fontsize=16, pad=20, fontweight="bold")
plt.ylabel("Descriptions", fontsize=12, labelpad=10)
plt.grid(True, axis="y", linestyle="--", alpha=0.3, color="gray")
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

A question we answer every day with this catalogue: *where can I find data about a topic, from which providers, and under which terms may I reuse it?* Public data on one subject is scattered across statistical offices, national portals and city portals, each with its own catalogue and its own words.

Here we look for datasets about air quality. We search the English and Italian descriptions for the topic's words, then join the catalogue to show each dataset's provider, title, links and licence. A simple word match is enough to start; a real search would build an index over `opendata_documents` with an embedding model of the reader's choice, since the catalogue ships no vectors.

In [ ]:
matches = con.sql(f"""
    SELECT c.provider, c.dataset_id, c.title, c.licence, c.sources
    FROM {table('opendata_documents')} d
    JOIN {table('opendata_catalog')} c USING (provider, dataset_id)
    WHERE (d.language = 'en' AND regexp_matches(lower(d.text), 'air quality|air pollution'))
       OR (d.language = 'it' AND regexp_matches(lower(d.text), 'qualità dell.aria|inquinamento atmosferico'))
    GROUP BY ALL
""").df()

print(len(matches), "datasets mention air quality")
matches.groupby("provider").size().sort_values(ascending=False)

In [ ]:
# The licences these datasets are published under
matches["licence"].fillna("not declared by the provider").value_counts().head(10)

In [ ]:
# Where to read three of them: the provider's own links
for row in matches.head(3).itertuples():
    print(row.provider, "|", row.title)
    pprint(json.loads(row.sources) if isinstance(row.sources, str) else row.sources)
    print()

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Which datasets of different providers measure the same thing for the same places?** ISTAT, Eurostat, the OECD and Italian cities often publish the same indicator (population, employment, waste, air quality) at different territorial levels, under different code lists and in different languages. The catalogue holds what is needed to link them: every dataset's dimensions (`opendata_structures`, `opendata_structure_dims`), every code with its labels in each language (`opendata_terms`), the codes that actually hold data (`opendata_held_codes`), and a searchable description per language (`opendata_documents`).

A crosswalk of equivalent datasets and code lists across providers would let anyone combine sources without reading every provider's catalogue. Some advice:

1. Start from the code lists, not the titles: territorial codes (NUTS, ISTAT municipality codes) and time periods are shared across providers and are a strong signal.
2. Compare labels across languages: `opendata_terms` names the same code in Italian, English, French, German and Spanish where providers publish them.
3. Use `opendata_held_codes` to keep only the combinations that hold data, so two datasets are linked only where they can actually be compared.
4. An open multilingual embedding model over `opendata_documents` is a good first pass for candidate pairs; the structures then confirm or reject them.